# COVID-19 French Charts
Guillaume Rozier, 2020

In [32]:
"""

LICENSE MIT
2020
Guillaume Rozier
Website : http://www.covidtracker.fr
Mail : guillaume.rozier@telecomnancy.net

README:
This file contains scripts that download data from data.gouv.fr and then process it to build many graphes.
I'm currently cleaning the code, please ask me if something is not clear enough.

The charts are exported to 'charts/images/france'.
Data is download to/imported from 'data/france'.
Requirements: please see the imports below (use pip3 to install them).

"""

"\n\nLICENSE MIT\n2020\nGuillaume Rozier\nWebsite : http://www.covidtracker.fr\nMail : guillaume.rozier@telecomnancy.net\n\nREADME:\nThis file contains scripts that download data from data.gouv.fr and then process it to build many graphes.\nI'm currently cleaning the code, please ask me if something is not clear enough.\n\nThe charts are exported to 'charts/images/france'.\nData is download to/imported from 'data/france'.\nRequirements: please see the imports below (use pip3 to install them).\n\n"

In [1]:
from multiprocessing import Pool
import requests
import pandas as pd
import math
import plotly.graph_objects as go
import plotly.express as px
import plotly
from plotly.subplots import make_subplots
from datetime import datetime
from datetime import timedelta
from tqdm import tqdm
import imageio
import json
import locale
import france_data_management as data
import numpy as np
import cv2

locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')
colors = px.colors.qualitative.D3 + plotly.colors.DEFAULT_PLOTLY_COLORS + px.colors.qualitative.Plotly + px.colors.qualitative.Dark24 + px.colors.qualitative.Alphabet
show_charts = False
PATH_STATS = "../../data/france/stats/"
PATH = "../../"
now = datetime.now()

In [2]:
try:
    #import subprocess
    #subprocess.run(["pip3", "install", "scikit-learn"])
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import PolynomialFeatures
    from sklearn.pipeline import make_pipeline

except:
    pass

# Data download and import

In [3]:
import time

success=False
tries = 0

while not success:
    try:
        data.download_data()
        success=True
    except Exception as e:
        print(e)
        time.sleep(20)
        print('retrying in 20s')
        tries += 1
        
        if tries >= 200:
            success=True
        continue

36it [00:04,  7.71it/s]                      


## Data transformations

In [36]:
df, df_confirmed, dates, df_new, df_tests, df_deconf, df_sursaud, df_incid, df_tests_viros = data.import_data()



  0%|          | 0/8 [00:00<?, ?it/s]

 38%|███▊      | 3/8 [00:04<00:07,  1.50s/it]

 75%|███████▌  | 6/8 [00:06<00:02,  1.26s/it]

21it [00:13,  1.02s/it]                      

28it [01:58,  5.23s/it]

36it [01:59,  3.67s/it]

In [37]:
df_incid_fra_clage = data.import_data_tests_sexe()
df_incid_fra = df_incid_fra_clage[df_incid_fra_clage["cl_age90"]==0]

In [38]:
df_new_france = df_new.groupby(["jour"]).sum().reset_index()

df_clage = data.import_data_hosp_clage()
df_clage_france = df_clage.groupby(["jour", "cl_age90"]).sum().reset_index()

df_incid = df_incid[df_incid["cl_age90"] == 0]

df_incid_france = df_incid.groupby("jour").sum().reset_index()
dates_clage = list(dict.fromkeys(list(df_clage_france['jour'].values))) 

df_sursaud_france = df_sursaud.groupby(['date_de_passage']).sum().reset_index()
df_sursaud_france["taux_covid"] = df_sursaud_france["nbre_pass_corona"] / df_sursaud_france["nbre_pass_tot"]
df_sursaud_france["taux_covid_acte"] = df_sursaud_france["nbre_acte_corona"] / df_sursaud_france["nbre_acte_tot"]
dates_sursaud = list(dict.fromkeys(list(df_sursaud['date_de_passage'].values))) 

dates_incid = list(dict.fromkeys(list(df_incid['jour'].values))) 
date_plus_1 = (datetime.strptime(dates_incid[-1], '%Y-%m-%d') + timedelta(days=2)).strftime('%Y-%m-%d')

departements = list(dict.fromkeys(list(df_incid['dep'].values))) 

last_day_plot = (datetime.strptime(max(dates), '%Y-%m-%d') + timedelta(days=1)).strftime("%Y-%m-%d")
last_day_plot_dashboard = (datetime.strptime(max(dates), '%Y-%m-%d') + timedelta(days=14)).strftime("%Y-%m-%d")

df_region = df.groupby(['regionName', 'jour', 'regionPopulation']).sum().reset_index()
df_region["hosp_regpop"] = df_region["hosp"] / df_region["regionPopulation"]*1000000 
df_region["rea_regpop"] = df_region["rea"] / df_region["regionPopulation"]*1000000 

df_tests_tot = df_tests.groupby(['jour']).sum().reset_index()

df_new_region = df_new.groupby(['regionName', 'jour']).sum().reset_index()
df_france = df.groupby('jour').sum().reset_index()

regions = list(dict.fromkeys(list(df['regionName'].values))) 
departements_noms = list(dict.fromkeys(list(df['departmentName'].values))) 

In [39]:
#Calcul sorties de réa
# Dataframe intermédiaire (décalée d'une ligne pour le calcul)
df_new_tot = df_new.groupby(["jour"]).sum().reset_index()
last_row = df_new_tot.iloc[-1]
df_new_tot = df_new_tot.shift()
df_new_tot = df_new_tot.append(last_row, ignore_index=True)

# Nouvelle dataframe contenant le résultat
df_new_tot["incid_dep_rea"] = df_france["rea"] - df_france["rea"].shift() - df_new_tot["incid_rea"]
df_new_tot["incid_dep_hosp_nonrea"] = df_france["hosp_nonrea"] - df_france["hosp_nonrea"].shift() - df_new_tot["incid_hosp_nonrea"].iloc[-1]
# On ne garde que les 19 derniers jours (rien d'intéressant avant)
df_new_tot_last15 = df_new_tot[ df_new_tot["jour"].isin(dates[:]) ]
df_france_last15 = df_france[ df_france["jour"].isin(dates[-19:]) ]
df_tests_tot_last15 = df_tests_tot[ df_tests_tot["jour"].isin(dates[-19:]) ]

In [40]:
departements_name = {}
for dep in departements:
    df_dep = df[df["dep"]==dep]["departmentName"]
    if len(df_dep):
        departements_name[dep] = df_dep.values[-1]
    else:
        departements_name[dep] = "na"
    if dep=="975":
        departements_name[dep] = "St-Pierre-et-Miquelon"

In [41]:
def objectif_deconfinement():
    dict_json = {}
    
    ## HOSP
    struct = {"dates": [], "values": []}
    n = 40
    dict_json["hosp"] = struct
    dict_json["hosp"]["values"] = [int(x) for x in df_france["hosp"].values[-n:]]
    dict_json["hosp"]["dates"] = list(df_france["jour"].values[-n:])
    
    ## REA
    struct = {"dates": [], "values": []}
    n = 40
    dict_json["rea"] = struct
    dict_json["rea"]["values"] = [int(x) for x in df_france["rea"].values[-n:]]
    dict_json["rea"]["dates"] = list(df_france["jour"].values[-n:])
    
    ## DC
    struct = {"dates": [], "values": []}
    n = 40
    dict_json["dc"] = struct
    dict_json["dc"]["values"] = [int(x) for x in df_france["dc"].diff().rolling(window=7).mean().values[-n:]]
    dict_json["dc"]["dates"] = list(df_france["jour"].values[-n:])
    
    ## Cas
    struct = {"date": "", "values": []}
    dict_json["cas"] = struct
    cas_rolling = df_incid_france["P"].rolling(window=7, center=False).mean().dropna()
    
    dict_json["cas"]["values"] = [int(x) for x in cas_rolling.values[-n:]]
    dict_json["cas"]["dates"] = list(df_incid_france.loc[cas_rolling.index.values[-n:], "jour"])

    with open(PATH_STATS + 'objectif_deconfinement.json', 'w') as outfile:
        json.dump(dict_json, outfile)
        
objectif_deconfinement()

In [42]:
def stats_immunite_collective(): 
    dict_json = {}
    
    coef_mortalite=[]
    for i in range(len(dates)):
        coef_mortalite += [0.007 - ((len(dates)-i))*0.0025/len(dates)]

    data_temp = df_france
    avant = data_temp[data_temp["jour"]=="2020-07-15"]["dc"].values[0]
    apres = data_temp["dc"].values[-1]

    dc_temp = data_temp["dc"].diff().values
    pop_immun_france = np.nan_to_num((dc_temp/coef_mortalite)).cumsum()/data_temp["departmentPopulation"].values[-1]*100
    dict_json["france"] = list(pop_immun_france)
    immun_deps=[]
    for dep in departements:
        data_temp = df[df["dep"]==dep]
        dc_temp = data_temp["dc"].diff().values
        if len(dc_temp):
            pop_immun = np.nan_to_num((dc_temp/coef_mortalite)).cumsum()/data_temp["departmentPopulation"].values[-1]*100
            dict_json[dep] = list(pop_immun)
            immun_deps += [list(pop_immun)]
            #print(dep, pop_immun[-1])
        else:
            continue
    dict_json["departements"] = departements
            
    with open(PATH_STATS + 'immunite.json', 'w') as outfile:
        json.dump(dict_json, outfile)
        
    return list(pop_immun_france), immun_deps
    
pop_immun_france, pop_immun_deps = stats_immunite_collective()

In [43]:
import random
df_temp = pd.DataFrame()
df_new_france["incid_rea"]

values_temp = []
dates_temp = []
for idx, death in enumerate(df_new_france["incid_dc"].rolling(window=7).mean().dropna().values):
    for point in range(int(death*1)):
        values_temp += [random.randrange(0, 10000, 1)/100]
        dates_temp += [df_new_france["jour"].values[idx+3]]
    

In [44]:
locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=dates_temp,
    y=values_temp,
    mode="markers",
    showlegend=False,
    marker_color="rgba(201, 4, 4,0.5)", #"rgba(0, 0, 0, 0.5)",
    marker_size=1.8))

fig.update_yaxes(range=[0, 100], visible=False)
fig.update_xaxes(tickformat="%d/%m", nticks=10)

fig.update_layout(
    plot_bgcolor='rgb(255,255,255)',
    title={
                'text': "Admissions en réanimation pour Covid19",
                'y':0.90,
                'x':0.5,
                'xanchor': 'center',
                'yanchor': 'top'},
                titlefont = dict(
                size=20),
    annotations = [
                dict(
                    x=0.5,
                    y=1.2,
                    xref='paper',
                    yref='paper',
                    text='Date : {}. Données : Santé publique France. Auteur : covidtracker.fr.'.format(datetime.strptime(max(dates), '%Y-%m-%d').strftime('%d %B %Y')),                    showarrow = False
                )]
                 
)
fig.write_image(PATH + "images/charts/france/points_deces.jpeg", scale=4, width=800, height=350)

In [45]:
clrs_sun_ref = ["#3c0000", "#4c0000", "#6a0000", "#840000", "#a00000", "#c40001", "#d50100", "#e20001", "#f50e07", "#f95228", "#fb9449", "#98ac3b", "#118408"]
clrs_sun_ref = ["#3c0000", "#840000", "#a00000", "#f50e07", "#f95228", "#118408"]
values_sun_ref = [300, 250, 200, 150, 100, 50]

incid = df_incid_fra["P"].rolling(window=7).sum().values/67114995*100000
incid = [incid[-i] for i in range(1, 50, 7)]
clrs_sun = ["white"]

for inc in incid:
    for i in range(len(values_sun_ref)):
        if inc>values_sun_ref[i]:
            clrs_sun += [clrs_sun_ref[i]]
            break
        
base="France"
fig =go.Figure(go.Sunburst(
    labels=[base, "Cette sem.", "S-1", "S-2", "S-3", "S-4", "S-5", "S-6"],
    parents=["", base, base, base, base, base, base, base],
    marker=dict(
        colors=clrs_sun,),
    values=[0.8, 4, 4, 4, 4, 4, 4, 4, 4, 4],
))
# Update layout for tight margin
# See https://plotly.com/python/creating-and-updating-figures/
fig.update_layout(margin = dict(t=0, l=0, r=0, b=0))

#fig.show()
